# Temporal LoRA Adapters

This notebooks trains temporal adapters on ECCO data.

We train adapters per decade (windom) and move each window with five year (step).

In [6]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
import json
import logging
import re
from datasets import Dataset
import pandas as pd
from pathlib import Path
from typing import Optional, Dict, Any

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("Using CPU - you will need to use a GPU to train models")



Using Apple MPS


In [ ]:
# Authenticate with Hugging Face (optional, for private models)
from huggingface_hub import login
login()  # Uncomment if you need to access private models

In [3]:
# Uncomment the line below to download the data from Google Drive using gdown
#!gdown 11wfdV7j1TBv_i9XOiT8G8V4NxnJTxezz

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [5]:
def load_csv_as_dataset(csv_paths):
    """Load CSV files and convert to Hugging Face Dataset."""
    all_data = []
    
    for csv_path in csv_paths:
        logger.info(f"Loading {Path(csv_path).name}...")
        df = pd.read_csv(csv_path)
        logger.info(f"  Rows: {len(df)}, Columns: {list(df.columns)}")
        all_data.append(df)
    
    combined_df = pd.concat(all_data, ignore_index=True)
    logger.info(f"Total rows: {len(combined_df)}")
    
    dataset = Dataset.from_pandas(combined_df)
    return dataset


In [8]:
base_path = '../data-processing-code/data'
csv_path = Path(base_path).glob('*_cleaned.csv')
dataset = load_csv_as_dataset(csv_path)

INFO:__main__:Loading ecco_pages_cleaned.csv...
INFO:__main__:  Rows: 169051, Columns: ['author', 'place', 'date', 'page_text', 'converted_date']
INFO:__main__:Total rows: 169051


In [15]:
adapters_windows = [(year, year+10) for year in list(range(1700,1795,5))]
adapters_windows[:2]

[(1700, 1710), (1705, 1715)]

In [ ]:


def preprocess_function(example):
    return {
        "text":  example["page_text"]
      }